# 02 - Compare a simple rule with a learned model
**Question:** Does learning from past examples beat copying last week's rentals?

Run notebook 01 first. We compare a last-week rule with one fixed gradient boosting model.
We choose using validation MAE, freeze that choice, then report test errors. We do not tune on test data.
The estimator is fitted only on training data; it is not refitted on validation data.


In [ ]:
from pathlib import Path
import sys
# Support opening the notebook from the project root or notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'mobility').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'mobility').is_dir():
    raise RuntimeError('Open Jupyter from the project folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from mobility.data import OUTPUT, invalidate_outputs
import pandas as pd
import numpy as np
import json
from IPython.display import display

import joblib
from sklearn.ensemble import HistGradientBoostingRegressor
from mobility.features import FEATURES, prediction_inputs
from mobility.evaluation import forecast_metrics
invalidate_outputs(2)
train = pd.read_csv(OUTPUT / 'train.csv')
validation = pd.read_csv(OUTPUT / 'validation.csv')
test = pd.read_csv(OUTPUT / 'test.csv')
x_train = prediction_inputs(train[FEATURES])
x_validation = prediction_inputs(validation[FEATURES])
x_test = prediction_inputs(test[FEATURES])
print('Train / validation / test rows:', len(train), len(validation), len(test))


## 1. Fit one model
The baseline needs no fitting: use `lag_168` directly.
Gradient boosting adds small decision trees in sequence, each helping correct previous errors.
`max_iter=150` limits the number of trees, `max_leaf_nodes=15` limits tree complexity, and
`learning_rate=0.08` controls each tree's contribution. These settings are fixed for this exercise.

`early_stopping=False` avoids the estimator making an internal random validation split.
Our separate, later time window handles validation. We clip predictions at zero because rentals cannot be negative.


In [ ]:
model = HistGradientBoostingRegressor(max_iter=150, max_leaf_nodes=15,
    learning_rate=0.08, min_samples_leaf=30, l2_regularization=1.0,
    early_stopping=False, random_state=42)
model.fit(x_train, train.cnt)
validation_predictions = {
    'last_week': validation.lag_168.to_numpy(),
    'gradient_boosting': np.maximum(0, model.predict(x_validation)),
}
validation_metrics = {name: forecast_metrics(validation.cnt, values)
                      for name, values in validation_predictions.items()}
# Dictionary order keeps the simpler rule if MAE is tied.
selected = min(validation_metrics, key=lambda name: validation_metrics[name]['mae'])
display(pd.DataFrame(validation_metrics).T.round(2))
print('Selected using validation MAE:', selected)


## 2. Freeze the choice, then evaluate test errors
MAE is average absolute error in rentals per observed hour. RMSE gives larger mistakes extra weight.
WAPE is total absolute error divided by total actual rentals, expressed as a percentage; it is not
"percent accuracy." Bias is total signed error divided by actual rentals: negative means underprediction.

The test window remains outside training and model selection. Reporting both test results is diagnostic;
we do not switch winners after looking at them. All metrics use the same eligible hours.


In [ ]:
artifact = {'selected_model': selected, 'features': FEATURES, 'estimator': model,
            'training_end': train.timestamp.max(), 'selection_metric': 'validation MAE'}
joblib.dump(artifact, OUTPUT / 'model.joblib')
test_predictions = {'last_week': test.lag_168.to_numpy(),
                    'gradient_boosting': np.maximum(0, model.predict(x_test))}
test_metrics = {name: forecast_metrics(test.cnt, values)
                for name, values in test_predictions.items()}
metrics = {'selected_model': selected, 'validation': validation_metrics, 'test': test_metrics,
           'test_mae_improvement_pct': 100 * (1-test_metrics[selected]['mae']/test_metrics['last_week']['mae'])}
(OUTPUT / 'metrics.json').write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding='utf-8')
for name, frame, predictions in [('validation', validation, validation_predictions),
                                  ('test', test, test_predictions)]:
    result = frame[['timestamp', 'hr', 'workingday', 'cnt']].copy()
    result['baseline_prediction'] = predictions['last_week']
    result['model_prediction'] = predictions['gradient_boosting']
    result['prediction'] = predictions[selected]
    result.to_csv(OUTPUT / f'{name}_predictions.csv', index=False)
display(pd.DataFrame(test_metrics).T.round(2))
print(f"Selected forecast test MAE improvement over last-week rule: {metrics['test_mae_improvement_pct']:.2f}%")


## What you should be able to explain
- Baseline: copy the actual recorded count from the same hour last week.
- Learned model: use calendar and two historical counts to estimate total rentals.
- Validation chooses; test measures the frozen choice.
- There are no guaranteed accuracy, fleet savings, or production claims.

**Next:** `03_capacity.ipynb` reads the saved prediction tables. Keep `model.joblib` local and only load trusted model files.
